<a href="https://colab.research.google.com/github/pngl2415/2.6-openMP/blob/main/Act_3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!nvcc --version

Wed Nov  4 11:00:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:

%%writefile operaciones_arreglos.cu
// PRACTICA COLABORATIVA 1.3
// REDUCCIONES COLECTIVAS EN GPU UTILIZANDO CUDA
//
// EQUIPO E01
// GUZMAN LOPEZ PAOLA NICOLE
// MONDRAGON GAMBINO JUAN SINUHE
// MORALES LIMON JENNIFER NATALY

#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <climits>
#include <random>
#include <stdexcept>
#include <iostream>
#include <limits>

// MODIFICAR
//constexpr int N = 40;
//constexpr int HILOS_POR_BLOQUE = 16;
//constexpr bool MOSTRAR_DETALLES = true;

constexpr int N = 4000000;
constexpr int HILOS_POR_BLOQUE = 256;
constexpr bool MOSTRAR_DETALLES = false;

constexpr const char* EQUIPO = "E01";
constexpr const char* INTEGRANTES =
    "GUZMAN LOPEZ PAOLA NICOLE | "
    "MONDRAGON GAMBINO JUAN SINUHE | "
    "MORALES LIMON JENNIFER NATALY";

// =====================================================
// COMPROBACION DE ERRORES CUDA
// =====================================================
void comprobarCUDA(cudaError_t error, const char* archivo, int linea) {
    if (error != cudaSuccess) {
        fprintf(stderr,
            "[HOST] [Equipo: %s] ERROR CUDA %s:%d: %s\n",
            EQUIPO, archivo, linea, cudaGetErrorString(error));
        throw std::runtime_error(cudaGetErrorString(error));
    }
}

#define CUDA_OK(x) comprobarCUDA((x), __FILE__, __LINE__)

// =====================================================
// KERNEL ARITMETICO
// Opciones 2, 3, 4 y 5
// =====================================================
__global__ void kernelAritmetico(
    const int* A, const int* B, long long* C,
    int n, int operacion, bool detalles
) {
    int globalIdx = blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx >= n) return;

    long long resultado = 0;

    switch (operacion) {
        case 2:
            resultado = (long long)A[globalIdx] + B[globalIdx];
            break;
        case 3:
            resultado = (long long)A[globalIdx] - B[globalIdx];
            break;
        case 4:
            resultado = (long long)A[globalIdx] * B[globalIdx];
            break;
        case 5:
            resultado = (long long)A[globalIdx] * A[globalIdx];
            break;
    }

    C[globalIdx] = resultado;

    if (detalles) {
        printf(
            "[DEVICE] [Equipo: E01] "
            "[Bloque: %d] [Hilo: %d] [Indice Global: %d] "
            "[Operacion: %d] A=%d B=%d Resultado=%lld\n",
            blockIdx.x, threadIdx.x, globalIdx,
            operacion, A[globalIdx], B[globalIdx], resultado
        );
    }
}

// =====================================================
// KERNEL DE REDUCCION PARALELA EN ARBOL BINARIO
// 0 = SUMATORIA
// 1 = MAXIMO
// 2 = MINIMO
// =====================================================
__global__ void kernelReduccion(
    const int* A, long long* parciales,
    int n, int modo, bool detalles
) {
    extern __shared__ long long compartida[];

    int tid = threadIdx.x;
    int globalIdx = blockIdx.x * blockDim.x * 2 + tid;

    long long neutro =
        modo == 1 ? LLONG_MIN :
        modo == 2 ? LLONG_MAX : 0LL;

    long long x =
        globalIdx < n ? (long long)A[globalIdx] : neutro;

    long long y =
        globalIdx + blockDim.x < n
        ? (long long)A[globalIdx + blockDim.x]
        : neutro;

    long long valor;

    if (modo == 0)
        valor = x + y;
    else if (modo == 1)
        valor = x > y ? x : y;
    else
        valor = x < y ? x : y;

    compartida[tid] = valor;

    if (detalles) {
        printf(
            "[DEVICE] [Equipo: E01] "
            "[Bloque: %d] [Hilo: %d] [Indice Global: %d] "
            "[Reduccion: %s] Valor local=%lld\n",
            blockIdx.x, tid, globalIdx,
            modo == 0 ? "SUMATORIA" :
            modo == 1 ? "MAXIMO" : "MINIMO",
            valor
        );
    }

    __syncthreads();

    for (int stride = blockDim.x / 2;
         stride > 0;
         stride >>= 1) {

        if (tid < stride) {
            long long otro = compartida[tid + stride];

            if (modo == 0)
                compartida[tid] += otro;
            else if (modo == 1)
                compartida[tid] =
                    compartida[tid] > otro
                    ? compartida[tid] : otro;
            else
                compartida[tid] =
                    compartida[tid] < otro
                    ? compartida[tid] : otro;
        }

        __syncthreads();
    }

    if (tid == 0) {
        parciales[blockIdx.x] = compartida[0];

        if (detalles) {
            printf(
                "[DEVICE] [Equipo: E01] "
                "[Bloque: %d] [Hilo: 0] "
                "[Indice Global: %d] "
                "Resultado parcial=%lld\n",
                blockIdx.x,
                blockIdx.x * blockDim.x * 2,
                compartida[0]
            );
        }
    }
}

// =====================================================
// CLASE OPERACIONESARREGLOS
// Integrantes:
// GUZMAN LOPEZ PAOLA NICOLE
// MONDRAGON GAMBINO JUAN SINUHE
// MORALES LIMON JENNIFER NATALY
// =====================================================
class OperacionesArreglos {
private:
    int* hA = nullptr;
    int* hB = nullptr;
    long long* hC = nullptr;

    int* dA = nullptr;
    int* dB = nullptr;
    long long* dC = nullptr;
    long long* dParciales = nullptr;

    int bloquesParciales = 0;

    bool creados = false;
    bool llenos = false;

    std::mt19937 generador{2026};

    void exigir(bool condicion, const char* mensaje) {
        if (!condicion)
            throw std::runtime_error(mensaje);
    }

    void copiarGPU() {
        CUDA_OK(cudaMemcpy(
            dA, hA, N * sizeof(int),
            cudaMemcpyHostToDevice
        ));

        CUDA_OK(cudaMemcpy(
            dB, hB, N * sizeof(int),
            cudaMemcpyHostToDevice
        ));
    }

    float ejecutarAritmetica(int opcion) {
        cudaEvent_t inicio = nullptr;
        cudaEvent_t fin = nullptr;

        CUDA_OK(cudaEventCreate(&inicio));

        try {
            CUDA_OK(cudaEventCreate(&fin));
        } catch (...) {
            cudaEventDestroy(inicio);
            throw;
        }

        float ms = 0;

        try {
            int bloques =
                (N + HILOS_POR_BLOQUE - 1) / HILOS_POR_BLOQUE;

            CUDA_OK(cudaEventRecord(inicio));

            kernelAritmetico<<<bloques, HILOS_POR_BLOQUE>>>(
                dA, dB, dC, N, opcion, MOSTRAR_DETALLES
            );

            CUDA_OK(cudaGetLastError());
            CUDA_OK(cudaEventRecord(fin));
            CUDA_OK(cudaEventSynchronize(fin));
            CUDA_OK(cudaEventElapsedTime(&ms, inicio, fin));

            CUDA_OK(cudaMemcpy(
                hC, dC, N * sizeof(long long),
                cudaMemcpyDeviceToHost
            ));

        } catch (...) {
            cudaEventDestroy(inicio);
            cudaEventDestroy(fin);
            throw;
        }

        CUDA_OK(cudaEventDestroy(inicio));
        CUDA_OK(cudaEventDestroy(fin));

        return ms;
    }

public:
    ~OperacionesArreglos() {
        liberar();
    }

    void liberar() noexcept {
        if (dParciales) cudaFree(dParciales);
        if (dC) cudaFree(dC);
        if (dB) cudaFree(dB);
        if (dA) cudaFree(dA);

        delete[] hC;
        delete[] hB;
        delete[] hA;

        dParciales = nullptr;
        dC = nullptr;
        dB = nullptr;
        dA = nullptr;

        hC = nullptr;
        hB = nullptr;
        hA = nullptr;

        bloquesParciales = 0;
        creados = false;
        llenos = false;
    }

    // OPCION 1
    void crearArregloCUDA() {
        liberar();

        try {
            hA = new int[N]();
            hB = new int[N]();
            hC = new long long[N]();

            CUDA_OK(cudaMalloc(
                (void**)&dA, N * sizeof(int)
            ));

            CUDA_OK(cudaMalloc(
                (void**)&dB, N * sizeof(int)
            ));

            CUDA_OK(cudaMalloc(
                (void**)&dC, N * sizeof(long long)
            ));

            bloquesParciales =
                (N + 2 * HILOS_POR_BLOQUE - 1) /
                (2 * HILOS_POR_BLOQUE);

            CUDA_OK(cudaMalloc(
                (void**)&dParciales,
                bloquesParciales * sizeof(long long)
            ));

            creados = true;

            printf(
                "[HOST] [Equipo: %s] Arreglos A, B y C creados\n"
                "[HOST] Elementos: %d\n"
                "[HOST] Hilos por bloque: %d\n"
                "[HOST] Bloques de reduccion: %d\n",
                EQUIPO, N, HILOS_POR_BLOQUE, bloquesParciales
            );

        } catch (...) {
            liberar();
            throw;
        }
    }

    // OPCION 6
    void llenarSecuencial() {
        exigir(creados, "Primero cree los arreglos");

        for (int i = 0; i < N; i++) {
            hA[i] = i + 1;
            hB[i] = i + 1;
        }

        copiarGPU();
        llenos = true;

        printf(
            "[HOST] [Equipo: %s] "
            "Llenado secuencial completado: 1 a %d\n",
            EQUIPO, N
        );
    }

    // OPCION 7
    void llenarAleatorio() {
        exigir(creados, "Primero cree los arreglos");

        std::uniform_int_distribution<int> dist(1, 1000000);

        for (int i = 0; i < N; i++) {
            hA[i] = dist(generador);
            hB[i] = dist(generador);
        }

        copiarGPU();
        llenos = true;

        printf(
            "[HOST] [Equipo: %s] "
            "Llenado aleatorio completado (1 a 1000000)\n",
            EQUIPO
        );
    }

    // OPCIONES 2, 3, 4 Y 5
    void aritmetica(int opcion) {
        exigir(llenos, "Primero cree y llene los arreglos");

        const char* nombre =
            opcion == 2 ? "SUMA" :
            opcion == 3 ? "RESTA" :
            opcion == 4 ? "MULTIPLICACION" : "CUADRADO";

        printf(
            "[HOST] [Equipo: %s] Iniciando %s en GPU\n",
            EQUIPO, nombre
        );

        float ms = ejecutarAritmetica(opcion);

        if (MOSTRAR_DETALLES) {
            for (int i = 0; i < N; i++) {
                printf(
                    "[HOST] [Equipo: %s] "
                    "[Indice Global: %d] C=%lld\n",
                    EQUIPO, i, hC[i]
                );
            }
        } else {
            printf(
                "[HOST] [Equipo: %s] "
                "Muestra: C[0]=%lld C[N-1]=%lld\n",
                EQUIPO, hC[0], hC[N - 1]
            );
        }

        printf(
            "[HOST] [Equipo: %s] "
            "Tiempo GPU (%s): %.4f ms\n",
            EQUIPO, nombre, ms
        );
    }

    // OPCIONES 8, 9, 10 Y 11
    void reduccion(int opcion) {
        exigir(llenos, "Primero cree y llene los arreglos");

        int modo =
            opcion == 10 ? 1 :
            opcion == 11 ? 2 : 0;

        const char* nombre =
            opcion == 8 ? "SUMATORIA" :
            opcion == 9 ? "PROMEDIO" :
            opcion == 10 ? "MAXIMO" : "MINIMO";

        printf(
            "[HOST] [Equipo: %s] "
            "Iniciando reduccion %s\n",
            EQUIPO, nombre
        );

        cudaEvent_t inicio = nullptr;
        cudaEvent_t fin = nullptr;

        CUDA_OK(cudaEventCreate(&inicio));

        try {
            CUDA_OK(cudaEventCreate(&fin));
        } catch (...) {
            cudaEventDestroy(inicio);
            throw;
        }

        float ms = 0;

        try {
            CUDA_OK(cudaEventRecord(inicio));

            kernelReduccion<<<
                bloquesParciales,
                HILOS_POR_BLOQUE,
                HILOS_POR_BLOQUE * sizeof(long long)
            >>>(
                dA, dParciales, N, modo, MOSTRAR_DETALLES
            );

            CUDA_OK(cudaGetLastError());
            CUDA_OK(cudaEventRecord(fin));
            CUDA_OK(cudaEventSynchronize(fin));
            CUDA_OK(cudaEventElapsedTime(&ms, inicio, fin));

            long long* parciales =
                new long long[bloquesParciales];

            try {
                CUDA_OK(cudaMemcpy(
                    parciales, dParciales,
                    bloquesParciales * sizeof(long long),
                    cudaMemcpyDeviceToHost
                ));

                long long global =
                    modo == 1 ? LLONG_MIN :
                    modo == 2 ? LLONG_MAX : 0LL;

                for (int i = 0; i < bloquesParciales; i++) {
                    if (modo == 0)
                        global += parciales[i];
                    else if (modo == 1 && parciales[i] > global)
                        global = parciales[i];
                    else if (modo == 2 && parciales[i] < global)
                        global = parciales[i];

                    if (MOSTRAR_DETALLES) {
                        printf(
                            "[HOST] [Equipo: %s] "
                            "[Bloque: %d] Parcial=%lld\n",
                            EQUIPO, i, parciales[i]
                        );
                    }
                }

                if (opcion == 9) {
                    printf(
                        "[HOST] [Equipo: %s] "
                        "Promedio global: %.6f\n",
                        EQUIPO, (double)global / N
                    );
                } else {
                    printf(
                        "[HOST] [Equipo: %s] "
                        "Resultado global (%s): %lld\n",
                        EQUIPO, nombre, global
                    );
                }

                printf(
                    "[HOST] [Equipo: %s] "
                    "Tiempo GPU (%s): %.4f ms (solo kernel)\n",
                    EQUIPO, nombre, ms
                );

            } catch (...) {
                delete[] parciales;
                throw;
            }

            delete[] parciales;

        } catch (...) {
            cudaEventDestroy(inicio);
            cudaEventDestroy(fin);
            throw;
        }

        CUDA_OK(cudaEventDestroy(inicio));
        CUDA_OK(cudaEventDestroy(fin));
    }
};

// =====================================================
// MENU INTERACTIVO - EXACTAMENTE 12 OPCIONES
// =====================================================
void mostrarMenu() {
    printf(
        "\n[HOST] [Equipo: E01] ===== MENU CUDA =====\n"
        "[HOST]  1. Crear arreglos\n"
        "[HOST]  2. Sumar arreglos\n"
        "[HOST]  3. Restar arreglos\n"
        "[HOST]  4. Multiplicar arreglos\n"
        "[HOST]  5. Calcular cuadrado de un arreglo\n"
        "[HOST]  6. Llenar secuencial\n"
        "[HOST]  7. Llenar aleatorio\n"
        "[HOST]  8. Sumatoria (Reduccion GPU)\n"
        "[HOST]  9. Promedio (Reduccion GPU)\n"
        "[HOST] 10. Maximo (Reduccion GPU)\n"
        "[HOST] 11. Minimo (Reduccion GPU)\n"
        "[HOST] 12. Salir\n"
        "[HOST] Seleccione opcion: "
    );

    fflush(stdout);
}

// =====================================================
// MAIN
// =====================================================
int main() {
    // PRIMERA IMPRESION: NOMBRES DEL EQUIPO
    printf(
        "[HOST] [Equipo: %s] INTEGRANTES: %s\n",
        EQUIPO, INTEGRANTES
    );

    int estado = 0;

    try {
        if (N <= 0 ||
            HILOS_POR_BLOQUE <= 0 ||
            HILOS_POR_BLOQUE > 1024 ||
            (HILOS_POR_BLOQUE &
             (HILOS_POR_BLOQUE - 1)) != 0) {
            throw std::runtime_error(
                "Configuracion invalida de N o hilos"
            );
        }

        int dispositivo = 0;
        cudaDeviceProp gpu{};

        CUDA_OK(cudaGetDevice(&dispositivo));
        CUDA_OK(cudaGetDeviceProperties(&gpu, dispositivo));

        printf(
            "[HOST] [Equipo: %s] GPU: %s\n"
            "[HOST] Multiprocesadores: %d\n"
            "[HOST] Maximo hilos por bloque: %d\n"
            "[HOST] Configuracion: N=%d, hilos/bloque=%d, detalle=%d\n",
            EQUIPO, gpu.name, gpu.multiProcessorCount,
            gpu.maxThreadsPerBlock,
            N, HILOS_POR_BLOQUE, (int)MOSTRAR_DETALLES
        );

        if (HILOS_POR_BLOQUE > gpu.maxThreadsPerBlock) {
            throw std::runtime_error(
                "Hilos por bloque exceden limite de GPU"
            );
        }

        OperacionesArreglos arreglos;

        int opcion;

        while (true) {
            mostrarMenu();

            if (!(std::cin >> opcion)) {
                if (std::cin.eof()) {
                    printf(
                        "\n[HOST] [Equipo: %s] "
                        "Fin de entrada\n", EQUIPO
                    );
                    break;
                }

                std::cin.clear();
                std::cin.ignore(
                    std::numeric_limits<std::streamsize>::max(),
                    '\n'
                );

                printf(
                    "[HOST] [Equipo: %s] Entrada invalida\n",
                    EQUIPO
                );
                continue;
            }

            if (opcion == 12) break;

            try {
                switch (opcion) {
                    case 1:
                        arreglos.crearArregloCUDA();
                        break;
                    case 2:
                    case 3:
                    case 4:
                    case 5:
                        arreglos.aritmetica(opcion);
                        break;
                    case 6:
                        arreglos.llenarSecuencial();
                        break;
                    case 7:
                        arreglos.llenarAleatorio();
                        break;
                    case 8:
                    case 9:
                    case 10:
                    case 11:
                        arreglos.reduccion(opcion);
                        break;
                    default:
                        printf(
                            "[HOST] [Equipo: %s] "
                            "Opcion invalida: %d\n",
                            EQUIPO, opcion
                        );
                }

            } catch (const std::exception& e) {
                printf(
                    "[HOST] [Equipo: %s] "
                    "ERROR CONTROLADO: %s\n",
                    EQUIPO, e.what()
                );
            }
        }

        printf(
            "[HOST] [Equipo: %s] "
            "Memoria liberada y programa finalizado\n",
            EQUIPO
        );

    } catch (const std::exception& e) {
        printf(
            "[HOST] [Equipo: %s] ERROR FATAL: %s\n",
            EQUIPO, e.what()
        );
        estado = 1;
    }

    // ULTIMA IMPRESION: NOMBRES DEL EQUIPO
    printf(
        "[HOST] [Equipo: %s] INTEGRANTES: %s\n",
        EQUIPO, INTEGRANTES
    );

    return estado;
}


Overwriting operaciones_arreglos.cu


In [13]:
!./operaciones_arreglos

[HOST] [Equipo: E01] INTEGRANTES: GUZMAN LOPEZ PAOLA NICOLE | MONDRAGON GAMBINO JUAN SINUHE | MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] GPU: Tesla T4
[HOST] Multiprocesadores: 40
[HOST] Maximo hilos por bloque: 1024
[HOST] Configuracion: N=4000000, hilos/bloque=256, detalle=0

[HOST] [Equipo: E01] ===== MENU CUDA =====
[HOST]  1. Crear arreglos
[HOST]  2. Sumar arreglos
[HOST]  3. Restar arreglos
[HOST]  4. Multiplicar arreglos
[HOST]  5. Calcular cuadrado de un arreglo
[HOST]  6. Llenar secuencial
[HOST]  7. Llenar aleatorio
[HOST]  8. Sumatoria (Reduccion GPU)
[HOST]  9. Promedio (Reduccion GPU)
[HOST] 10. Maximo (Reduccion GPU)
[HOST] 11. Minimo (Reduccion GPU)
[HOST] 12. Salir
[HOST] Seleccione opcion: 1
[HOST] [Equipo: E01] Arreglos A, B y C creados
[HOST] Elementos: 4000000
[HOST] Hilos por bloque: 256
[HOST] Bloques de reduccion: 7813

[HOST] [Equipo: E01] ===== MENU CUDA =====
[HOST]  1. Crear arreglos
[HOST]  2. Sumar arreglos
[HOST]  3. Restar arreglos
[HOST]  4. Mul

con 40 elementos


In [10]:
!nvcc -std=c++17 -O2 operaciones_arreglos.cu -o operaciones_arreglos

con 4 millones de elementos

In [11]:
!cp operaciones_arreglos.cu operaciones_arreglos_40_respaldo.cu

In [12]:
!printf '1\n7\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos 2>&1 | tee -a log_ejecucion_cuda.txt

[HOST] [Equipo: E01] INTEGRANTES: GUZMAN LOPEZ PAOLA NICOLE | MONDRAGON GAMBINO JUAN SINUHE | MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] GPU: Tesla T4
[HOST] Multiprocesadores: 40
[HOST] Maximo hilos por bloque: 1024
[HOST] Configuracion: N=4000000, hilos/bloque=256, detalle=0

[HOST] [Equipo: E01] ===== MENU CUDA =====
[HOST]  1. Crear arreglos
[HOST]  2. Sumar arreglos
[HOST]  3. Restar arreglos
[HOST]  4. Multiplicar arreglos
[HOST]  5. Calcular cuadrado de un arreglo
[HOST]  6. Llenar secuencial
[HOST]  7. Llenar aleatorio
[HOST]  8. Sumatoria (Reduccion GPU)
[HOST]  9. Promedio (Reduccion GPU)
[HOST] 10. Maximo (Reduccion GPU)
[HOST] 11. Minimo (Reduccion GPU)
[HOST] 12. Salir
[HOST] Seleccione opcion: [HOST] [Equipo: E01] Arreglos A, B y C creados
[HOST] Elementos: 4000000
[HOST] Hilos por bloque: 256
[HOST] Bloques de reduccion: 7813

[HOST] [Equipo: E01] ===== MENU CUDA =====
[HOST]  1. Crear arreglos
[HOST]  2. Sumar arreglos
[HOST]  3. Restar arreglos
[HOST]  4. Multi

In [8]:
!ls -lh log_ejecucion_cuda.txt
!grep -E 'GPU:|Configuracion:|Resultado global|Promedio global|Tiempo GPU|INTEGRANTES:' log_ejecucion_cuda.txt

-rw-r--r-- 1 root root 66K Nov  4 12:24 log_ejecucion_cuda.txt
[HOST] [Equipo: E01] INTEGRANTES: GUZMAN LOPEZ PAOLA NICOLE | MONDRAGON GAMBINO JUAN SINUHE | MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] GPU: Tesla T4
[HOST] Configuracion: N=40, hilos/bloque=16, detalle=1
[HOST] [Equipo: E01] Tiempo GPU (SUMA): 1.7364 ms
[HOST] [Equipo: E01] Tiempo GPU (RESTA): 0.0825 ms
[HOST] [Equipo: E01] Tiempo GPU (MULTIPLICACION): 0.0825 ms
[HOST] [Equipo: E01] Tiempo GPU (CUADRADO): 0.0792 ms
[HOST] [Equipo: E01] Resultado global (SUMATORIA): 820
[HOST] [Equipo: E01] Tiempo GPU (SUMATORIA): 0.1581 ms (solo kernel)
[HOST] [Equipo: E01] Promedio global: 20.500000
[HOST] [Equipo: E01] Tiempo GPU (PROMEDIO): 0.1626 ms (solo kernel)
[HOST] [Equipo: E01] Resultado global (MAXIMO): 40
[HOST] [Equipo: E01] Tiempo GPU (MAXIMO): 0.1340 ms (solo kernel)
[HOST] [Equipo: E01] Resultado global (MINIMO): 1
[HOST] [Equipo: E01] Tiempo GPU (MINIMO): 0.1312 ms (solo kernel)
[HOST] [Equipo: E01] Resultado glob

# **Ejecutar con 40 elementos**

In [3]:
!printf '1\n6\n2\n3\n4\n5\n8\n9\n10\n11\n7\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos 2>&1 | tee log_ejecucion_cuda.txt

[HOST] [Equipo: E01] INTEGRANTES: GUZMAN LOPEZ PAOLA NICOLE | MONDRAGON GAMBINO JUAN SINUHE | MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] GPU: Tesla T4
[HOST] Multiprocesadores: 40
[HOST] Maximo hilos por bloque: 1024
[HOST] Configuracion: N=40, hilos/bloque=16, detalle=1

[HOST] [Equipo: E01] ===== MENU CUDA =====
[HOST]  1. Crear arreglos
[HOST]  2. Sumar arreglos
[HOST]  3. Restar arreglos
[HOST]  4. Multiplicar arreglos
[HOST]  5. Calcular cuadrado de un arreglo
[HOST]  6. Llenar secuencial
[HOST]  7. Llenar aleatorio
[HOST]  8. Sumatoria (Reduccion GPU)
[HOST]  9. Promedio (Reduccion GPU)
[HOST] 10. Maximo (Reduccion GPU)
[HOST] 11. Minimo (Reduccion GPU)
[HOST] 12. Salir
[HOST] Seleccione opcion: [HOST] [Equipo: E01] Arreglos A, B y C creados
[HOST] Elementos: 40
[HOST] Hilos por bloque: 16
[HOST] Bloques de reduccion: 2

[HOST] [Equipo: E01] ===== MENU CUDA =====
[HOST]  1. Crear arreglos
[HOST]  2. Sumar arreglos
[HOST]  3. Restar arreglos
[HOST]  4. Multiplicar arreglos

In [19]:
from pathlib import Path

codigo = Path("operaciones_arreglos.cu").read_text()

codigo = codigo.replace(
    "constexpr int N = 4000000;",
    "constexpr int N = 40;"
)
codigo = codigo.replace(
    "constexpr int HILOS_POR_BLOQUE = 256;",
    "constexpr int HILOS_POR_BLOQUE = 16;"
)
codigo = codigo.replace(
    "constexpr bool MOSTRAR_DETALLES = false;",
    "constexpr bool MOSTRAR_DETALLES = true;"
)

Path("operaciones_arreglos_40.cu").write_text(codigo)

print("Archivo de 40 elementos creado correctamente.")

Archivo de 40 elementos creado correctamente.


# **Ejecutar con 4 millones de elementos**

In [5]:
!printf '1\n7\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos | tee log_ejecucion_cuda_masiva.txt

[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Hilos maximos por bloque: 1024
[HOST] [Equipo: E01] Memoria global: 15637086208 bytes
[HOST] [Equipo: E01] Memoria compartida por bloque: 49152 bytes
[HOST] [Equipo: E01] Configuracion: N=4000000, hilos/bloque=256, detalle=0

[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 2. Sumar arreglos
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E01] 9. Promedio GPU
[HOST] [Equipo: E01] 10. Maximo G

# **Cuatro reducciones: sumatoria, promedio, máximo y mínimo.**

In [6]:
!grep -E 'Configuracion:|Resultado global|Promedio global|Tiempo GPU' log_ejecucion_cuda_masiva.txt

[HOST] [Equipo: E01] Configuracion: N=4000000, hilos/bloque=256, detalle=0
[HOST] [Equipo: E01] Resultado global (Sumatoria): 1999994687964
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 2.9479 ms
[HOST] [Equipo: E01] Resultado global (Sumatoria): 1999994687964
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 0.1763 ms
[HOST] [Equipo: E01] Promedio global: 499998.671991
[HOST] [Equipo: E01] Resultado global (Maximo): 1000000
[HOST] [Equipo: E01] Tiempo GPU (Maximo): 0.1946 ms
[HOST] [Equipo: E01] Resultado global (Minimo): 1
[HOST] [Equipo: E01] Tiempo GPU (Minimo): 0.1866 ms


In [18]:
!nvcc -std=c++17 -O2 operaciones_arreglos_40.cu -o operaciones_arreglos_40

# **Para ejecutar todas las operaciones**

In [10]:
!printf '1\n6\n2\n3\n4\n5\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos_40 | tee log_ejecucion_cuda_40.txt

[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Hilos maximos por bloque: 1024
[HOST] [Equipo: E01] Memoria global: 15637086208 bytes
[HOST] [Equipo: E01] Memoria compartida por bloque: 49152 bytes
[HOST] [Equipo: E01] Configuracion: N=40, hilos/bloque=16, detalle=1

[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 2. Sumar arreglos
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E01] 9. Promedio GPU
[HOST] [Equipo: E01] 10. Maximo GPU
[HO

# **Mostrar las reducciones de los resultados principales**

In [11]:
!grep -E 'Configuracion:|Resultado global|Promedio global|Tiempo GPU' log_ejecucion_cuda_40.txt

[HOST] [Equipo: E01] Configuracion: N=40, hilos/bloque=16, detalle=1
[HOST] [Equipo: E01] Tiempo GPU (Suma): 1.8201 ms
[HOST] [Equipo: E01] Tiempo GPU (Resta): 0.1026 ms
[HOST] [Equipo: E01] Tiempo GPU (Multiplicacion): 0.1027 ms
[HOST] [Equipo: E01] Tiempo GPU (Cuadrado): 0.1070 ms
[HOST] [Equipo: E01] Resultado global (Sumatoria): 820
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 0.1983 ms
[HOST] [Equipo: E01] Resultado global (Sumatoria): 820
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 0.1752 ms
[HOST] [Equipo: E01] Promedio global: 20.500000
[HOST] [Equipo: E01] Resultado global (Maximo): 40
[HOST] [Equipo: E01] Tiempo GPU (Maximo): 0.1874 ms
[HOST] [Equipo: E01] Resultado global (Minimo): 1
[HOST] [Equipo: E01] Tiempo GPU (Minimo): 0.1868 ms


Registro de las dos ejecuciones

In [12]:
!cp log_ejecucion_cuda_masiva.txt log_ejecucion_cuda.txt

Verificar el archivo, sus primeras y últimas líneas

In [13]:
!ls -lh log_ejecucion_cuda.txt
!head -n 5 log_ejecucion_cuda.txt
!tail -n 5 log_ejecucion_cuda.txt

-rw-r--r-- 1 root root 5.8K Nov  4 11:12 log_ejecucion_cuda.txt
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Seleccione opcion: 
[HOST] [Equipo: E01] Opcion seleccionada: 12
[HOST] [Equipo: E01] Saliendo del programa.
[HOST] [Equipo: E01] Memoria liberada correctamente.
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY


In [14]:
import subprocess

programa = "./operaciones_arreglos_40"

print("=== MENU INTERACTIVO CUDA - EQUIPO E01 ===")
print("1. Crear arreglos")
print("2. Sumar arreglos")
print("3. Restar arreglos")
print("4. Multiplicar arreglos")
print("5. Calcular cuadrado")
print("6. Llenar secuencial")
print("7. Llenar aleatorio")
print("8. Sumatoria GPU")
print("9. Promedio GPU")
print("10. Maximo GPU")
print("11. Minimo GPU")
print("12. Salir")

opciones = []

while True:
    opcion = input("\nSeleccione una opcion (1-12): ").strip()

    if not opcion.isdigit() or not 1 <= int(opcion) <= 12:
        print("Opcion no valida.")
        continue

    opciones.append(opcion)

    if opcion == "12":
        break

    print("Opcion registrada:", opcion)

print("\nEjecutando operaciones seleccionadas...\n")

resultado = subprocess.run(
    [programa],
    input="\n".join(opciones) + "\n",
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.stderr:
    print(resultado.stderr)

=== MENU INTERACTIVO CUDA - EQUIPO E01 ===
1. Crear arreglos
2. Sumar arreglos
3. Restar arreglos
4. Multiplicar arreglos
5. Calcular cuadrado
6. Llenar secuencial
7. Llenar aleatorio
8. Sumatoria GPU
9. Promedio GPU
10. Maximo GPU
11. Minimo GPU
12. Salir

Seleccione una opcion (1-12): 1
Opcion registrada: 1

Seleccione una opcion (1-12): 2
Opcion registrada: 2

Seleccione una opcion (1-12): 3
Opcion registrada: 3

Seleccione una opcion (1-12): 4
Opcion registrada: 4

Seleccione una opcion (1-12): 5
Opcion registrada: 5

Seleccione una opcion (1-12): 6
Opcion registrada: 6

Seleccione una opcion (1-12): 7
Opcion registrada: 7

Seleccione una opcion (1-12): 8
Opcion registrada: 8

Seleccione una opcion (1-12): 9
Opcion registrada: 9

Seleccione una opcion (1-12): 10
Opcion registrada: 10

Seleccione una opcion (1-12): 11
Opcion registrada: 11

Seleccione una opcion (1-12): 12

Ejecutando operaciones seleccionadas...

[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGO

In [20]:
!./operaciones_arreglos_40

[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Hilos maximos por bloque: 1024
[HOST] [Equipo: E01] Memoria global: 15637086208 bytes
[HOST] [Equipo: E01] Memoria compartida por bloque: 49152 bytes
[HOST] [Equipo: E01] Configuracion: N=40, hilos/bloque=16, detalle=1

[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 2. Sumar arreglos
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E01] 9. Promedio GPU
[HOST] [Equipo: E01] 10. Maximo GPU
[HO